### Import Libraries

In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))


In [ ]:
import torch
from torch import nn

from transformers import WhisperProcessor, WhisperForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

from src.model import WhisperAccentConfig, WhisperAccentForConditionalGeneration, WhisperAccentProcessor, register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import processor_init, model_init, ModelArguments, LoraArguments, WhisperAccentTrainingArguments
from src.train.trainer import WhisperAccentTrainer
from src.model.tokenization import ACCENTS
from src.utils.loading import load_model_from_pretrained

register_whisper_accent()


### Create Arguments

In [ ]:
MODEL_TYPE = "whisper_accent"
BASE_MODEL_NAME_OR_PATH = "openai/whisper-small.en"
IS_MULTILINGUAL = False
DATASET_NAME="westbrook/English_Accent_DataSet"

model_args = ModelArguments(
    model_type=MODEL_TYPE,
    base_model_name_or_path=BASE_MODEL_NAME_OR_PATH,
    is_multilingual=IS_MULTILINGUAL,
)

lora_args = LoraArguments(
    lora_enable=True,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.15,
    lora_bias="none",
    use_rslora=True,
    task_type="SEQ_2_SEQ_LM",
)

training_args = WhisperAccentTrainingArguments(
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    lambda_accent_loss=0.0,
    lambda_diversity_loss=0.0,
    optim="adamw_torch",
    learning_rate=1e-5,
    embedding_learning_rate=5e-5,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=0.05,
    max_steps=100,
    max_grad_norm=1.0,
    eval_strategy="steps",
    eval_steps=50,
    eval_on_start=True,
    predict_with_generate=True,
    logging_first_step=True,
    logging_steps=10,
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)


### Load Model and Processor

In [ ]:
# Load model and processor; whisper / whisper_accent models are supported
if model_args.model_type == "whisper_accent":
    processor = processor_init(model_args.base_model_name_or_path)
    model = model_init(model_args.base_model_name_or_path, processor)
elif model_args.model_type == "whisper":
    processor = WhisperProcessor.from_pretrained(model_args.base_model_name_or_path)
    model = WhisperForConditionalGeneration.from_pretrained(model_args.base_model_name_or_path)
    # Update generation config; https://github.com/openai/whisper/discussions/2094
    if model.generation_config.is_multilingual:
        model.generation_config.language = "en"
        model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = None
else:
    raise ValueError(f"Invalid model type: {model_args.model_type}")

# Add LoRA layers
# Note: Non-LoRA training is not implemented yet
if lora_args.lora_enable:
    # Target linear layers
    target_modules = []
    m_list = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]
    for name, _ in model.named_modules():
        if any(suffix in name for suffix in m_list):
            target_modules.append(name)

    lora_config = LoraConfig(
        r=lora_args.lora_r,
        lora_alpha=lora_args.lora_alpha,
        lora_dropout=lora_args.lora_dropout,
        bias=lora_args.lora_bias,
        use_rslora=lora_args.use_rslora,
        target_modules=target_modules,
        task_type=lora_args.task_type,
        ensure_weight_tying=True,
    )

    # Trainable token indices for new accent tokens
    if model_args.model_type == "whisper_accent":
        accent_token_indices = sorted(list(model.generation_config.accent_to_id.values()))
        lora_config.trainable_token_indices = accent_token_indices

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    raise NotImplementedError("Non-LoRA training is not implemented yet")


### Training

In [ ]:
import datetime

run_id = "whisper-test-run" 
run_name = f"{run_id}-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
output_dir = f"/workspace/checkpoints/{run_id}"


In [ ]:
training_args.set_save(
    strategy="steps",
    steps=50,
    total_limit=100,
)
training_args.set_push_to_hub(
    model_id=f"mavleo96/{run_id}",
    strategy="all_checkpoints",
)
training_args.output_dir = output_dir


In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

train_dataset = WhisperDataset(
    data_path=DATASET_NAME,
    split="train",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path=DATASET_NAME,
    split="validation",
    processor=processor,
    multilingual_model=model_args.is_multilingual,
    num_proc=16,
)
eval_dataset.raw_dataset = eval_dataset.raw_dataset.select(range(20))


In [ ]:
trainer = WhisperAccentTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    compute_metrics="all" if model_args.model_type == "whisper_accent" else "wer",
)

In [ ]:
trainer.train()
